In [ ]:
from os import listdir
from experiment_main import *

def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    )


def return_strategy_performance(
    nikkei_data,
    major_list,
    start_timestamp,
    data_part,
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    two_years = 24
    start_optimize = make_new_time(start_timestamp, -two_years)
    end_optimize = start_timestamp
    optimize_data = nikkei_data[start_optimize:end_optimize]

    sys_data = [
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
        [0 for i in range(14)],
    ]

    name_mode = get_name_mode(strategy)
    (
        dff,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ) = get_crossover_results(
        optimize_data,
        data_part,
        name_mode,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade,
        alpha_range,
        gamma_range,
        signal_line_if,
        signal_line_if_target,
        last_close,
    )

    add_df_to_list_detailed(
        major_list,
        dff,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    )

    add_df_to_list_detailed(
        sys_data,
        dff,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    )
    for i in range(len(sys_data)):
        sys_data[i].extend(sys_data[i][-3:])
        print(sys_data[i], len(sys_data[i]))

    best_sys_minor = get_best_results(sys_data)

    return (
        dff,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
        major_list,
        sys_data,
        best_sys_minor,
    )


def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # (rule_data[i][11]), # / ticker_length)
                # rule_data[i][12],
                # (rule_data[i][13] * 100 / rule_data[i][12]) if rule_data[i][12] != 0 else 0,
                # (rule_data[i][14]), # / ticker_length)
                # rule_data[i][15],
                # (rule_data[i][16] * 100 / rule_data[i][15]) if rule_data[i][15] != 0 else 0,
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]


def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            # "Long 1 Return",
            # "Long 1 Transactions",
            # "Long 1 Win Ratio",
            # "Long 2 Return",
            # "Long 2 Transactions",
            # "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)

In [2]:
market = "nikkei"
time_format = "1_day"

macd_params = (12, 26, 9)
# macd_params = (4, 22, 3)


params_str = "_".join(str(val) for val in macd_params)
short_period, long_period, signal_period = macd_params

trade_cost = False
balance = 0
risk_per_trade = 100 / 100
last_close = True
strategy = "MACDCROSS"
# strategy = "ZEROCROSS"
target_list = histogram_list if strategy == "MACDCROSS" else macd_list

prefix = f"./data/{market}/"

nikkei_data = pd.read_csv(
    prefix + "nkf.csv",
    index_col="Date",
    parse_dates=True,
    sep=";",
    usecols=["Date", "Close", "Open", "High", "Low"],
)

nikkei_data.dropna(inplace=True)
nikkei_data.columns = list(map(lambda x: x.lower(), list(nikkei_data.columns)))
nikkei_data.sort_index(inplace=True)
nikkei_data.index.names = ["date"]

nikkei_data = calculate_indicators(
    nikkei_data,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
)

nikkei_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2009-07-01,10000.0,10100.0,9982.0,10047.0,128.330393,137.438745,-9.108352,146.992133,138.450280,8.541853,...,-20.199978,3285,164.965806,185.165784,-20.199978,3288,164.965806,185.165784,-20.199978,3289
2009-07-02,9878.0,9913.0,9707.0,9738.0,107.842157,131.519427,-23.677270,72.607180,125.281660,-52.674480,...,-19.090816,3285,161.302265,180.393080,-19.090816,3288,161.302265,180.393080,-19.090816,3289
2009-07-03,9843.0,9858.0,9795.0,9803.0,95.746336,124.364809,-28.618473,116.571576,123.539643,-6.968067,...,-19.549461,3285,155.956254,175.505715,-19.549461,3288,155.956254,175.505715,-19.549461,3289
2009-07-06,9680.0,9748.0,9640.0,9727.0,79.115748,115.314997,-36.199249,45.405502,107.912815,-62.507313,...,-20.456362,3285,149.935262,170.391624,-20.456362,3288,149.935262,170.391624,-20.456362,3289
2009-07-07,9668.0,9727.0,9527.0,9552.0,51.224367,102.496871,-51.272504,56.461345,97.622521,-41.161176,...,-22.030651,3285,142.853311,164.883962,-22.030651,3288,142.853311,164.883962,-22.030651,3289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,23825.0,23830.0,23755.0,23780.0,199.353793,206.105686,-6.751893,208.396228,206.348214,2.048014,...,-0.776993,3285,208.764901,209.541894,-0.776993,3288,208.764901,209.541894,-0.776993,3289
2019-12-26,23770.0,23960.0,23725.0,23955.0,200.757806,205.036110,-4.278304,193.883226,203.855217,-9.971991,...,-1.374483,3285,207.823790,209.198273,-1.374483,3288,207.823790,209.198273,-1.374483,3289
2019-12-27,23910.0,23960.0,23770.0,23795.0,186.806445,201.390177,-14.583732,180.437342,199.171642,-18.734300,...,-1.664865,3285,207.117192,208.782057,-1.664865,3288,207.117192,208.782057,-1.664865,3289
2019-12-30,23830.0,23830.0,23415.0,23430.0,144.630234,190.038189,-45.407955,112.403837,181.818081,-69.414244,...,-2.956752,3285,205.086117,208.042869,-2.956752,3288,205.086117,208.042869,-2.956752,3289


In [3]:
two_years = 24
# zero_check = pd.Timestamp("2008-01-01")
first_check = pd.Timestamp("2011-01-01")
second_check = pd.Timestamp("2014-01-01")
third_check = pd.Timestamp("2017-01-01")
fourth_check = pd.Timestamp("2020-01-01")

# checks_list = [zero_check, first_check, second_check, third_check, fourth_check]
checks_list = [first_check, second_check, third_check, fourth_check]

# zero_part = nikkei_data[zero_check:first_check]
first_part = nikkei_data[first_check:second_check]
second_part = nikkei_data[second_check:third_check]
third_part = nikkei_data[third_check:fourth_check]

# df_list = [zero_part, first_part, second_part, third_part]
df_list = [first_part, second_part, third_part]

first_part

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2011-01-04,10375.0,10460.0,10345.0,10345.0,86.478559,111.111833,-24.633274,97.119975,109.630343,-12.510367,...,-8.661793,3285,128.097349,136.759107,-8.661758,3288,128.097349,136.759107,-8.661758,3289
2011-01-05,10420.0,10555.0,10360.0,10545.0,101.376612,109.164789,-7.788177,105.207585,108.745791,-3.538206,...,-10.258937,3285,123.935470,134.194380,-10.258910,3288,123.935470,134.194380,-10.258910,3289
2011-01-06,10540.0,10585.0,10495.0,10520.0,109.899285,109.311688,0.587597,114.121814,109.820996,4.300819,...,-10.011859,3285,121.679585,131.691421,-10.011836,3288,121.679585,131.691421,-10.011836,3289
2011-01-07,10545.0,10595.0,10455.0,10505.0,114.127598,110.274870,3.852728,114.132803,110.683357,3.449446,...,-8.951911,3285,120.501555,129.453448,-8.951893,3288,120.501555,129.453448,-8.951893,3289
2011-01-10,10545.0,10545.0,10400.0,10480.0,114.145477,111.048992,3.096485,114.156884,111.378063,2.778821,...,-7.671445,3285,119.864159,127.535590,-7.671431,3288,119.864159,127.535590,-7.671431,3289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-24,16047.5,16050.5,15863.0,15882.5,215.520947,198.510184,17.010763,239.813533,202.883480,36.930053,...,-6.063491,3285,201.836857,207.900347,-6.063491,3288,201.836857,207.900347,-6.063491,3289
2013-12-26,16212.5,16251.5,16197.5,16251.5,243.571294,207.522406,36.048888,246.953278,211.697439,35.255839,...,-3.756065,3285,203.205266,206.961331,-3.756065,3288,203.205266,206.961331,-3.756065,3289
2013-12-27,16247.5,16312.5,16063.0,16292.5,266.042975,219.226520,46.816455,283.223702,226.002692,57.221010,...,0.224430,3285,207.241868,207.017439,0.224430,3288,207.241868,207.017439,0.224430,3289
2013-12-30,16290.5,16333.0,16183.0,16291.5,280.537392,231.488694,49.048698,278.119713,236.426096,41.693617,...,4.883632,3285,213.121979,208.238347,4.883632,3288,213.121979,208.238347,4.883632,3289


# SYSTEM 1 

In [4]:
signal_line_if = False
signal_line_if_target = False
sys1_major = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]

# first part

In [5]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys1_major,
    sys1_first,
    best_sys1_first,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys1_major,
    first_check,
    first_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[2912.5, 56.00961538461539, 0, 52, 22, 26, 13, 4157.5, 26, 9, -1245.0, 4157.5, 26, 13, 4157.5, 26, 13] 17
[1097.5, 22.864583333333332, 0, 48, 18, 24, 10, 3197.5, 24, 8, -2100.0, 3197.5, 24, 10, 3197.5, 24, 10] 17
[3259.5, 60.361111111111114, 0, 54, 24, 27, 14, 4411.5, 27, 10, -1152.0, 4411.5, 27, 14, 4411.5, 27, 14] 17
[3259.5, 60.361111111111114, 0, 54, 24, 27, 14, 4411.5, 27, 10, -1152.0, 4411.5, 27, 14, 4411.5, 27, 14] 17
[2015.5, 33.59166666666667, 0, 60, 21, 30, 14, 3710.5, 30, 7, -1695.0, 3710.5, 30, 14, 3710.5, 30, 14] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
283,2912.5,56.009615,0,52,22,26,13,4157.5,26,9,-1245.0,4157.5,26,13,Histogram a: 0.2 g: 1
15,1097.5,22.864583,0,48,18,24,10,3197.5,24,8,-2100.0,3197.5,24,10,Histogram a: 0.6
263,3259.5,60.361111,0,54,24,27,14,4411.5,27,10,-1152.0,4411.5,27,14,Histogram a: 0.1 g: 20
275,3259.5,60.361111,0,54,24,27,14,4411.5,27,10,-1152.0,4411.5,27,14,Histogram a: 0.1 g: 500
9,2015.5,33.591667,0,60,21,30,14,3710.5,30,7,-1695.0,3710.5,30,14,Histogram a: 0


# second part

In [6]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys1_major,
    sys1_second,
    best_sys1_second,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys1_major,
    second_check,
    second_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-3523.5, -53.38636363636363, 0, 66, 17, 33, 12, -21.5, 33, 5, -3502.0, -21.5, 33, 12, -21.5, 33, 12] 17
[-801.5, -10.01875, 0, 80, 22, 40, 12, 1380.5, 40, 10, -2182.0, 1380.5, 40, 12, 1380.5, 40, 12] 17
[-801.5, -10.01875, 0, 80, 22, 40, 12, 1380.5, 40, 10, -2182.0, 1380.5, 40, 12, 1380.5, 40, 12] 17
[-3523.5, -53.38636363636363, 0, 66, 17, 33, 12, -21.5, 33, 5, -3502.0, -21.5, 33, 12, -21.5, 33, 12] 17
[-2481.5, -40.024193548387096, 0, 62, 19, 31, 12, 545.5, 31, 7, -3027.0, 545.5, 31, 12, 545.5, 31, 12] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
213,-3523.5,-53.386364,0,66,17,33,12,-21.5,33,5,-3502.0,-21.5,33,12,Histogram a: -0.1 g: 4
3,-801.5,-10.018750,0,80,22,40,12,1380.5,40,10,-2182.0,1380.5,40,12,Histogram a: -0.6
3,-801.5,-10.018750,0,80,22,40,12,1380.5,40,10,-2182.0,1380.5,40,12,Histogram a: -0.6
214,-3523.5,-53.386364,0,66,17,33,12,-21.5,33,5,-3502.0,-21.5,33,12,Histogram a: -0.1 g: 10
9,-2481.5,-40.024194,0,62,19,31,12,545.5,31,7,-3027.0,545.5,31,12,Histogram a: 0


# third part

In [7]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys1_major,
    sys1_third,
    best_sys1_third,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys1_major,
    third_check,
    third_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[-400.0, -6.779661016949152, 0, 59, 19, 30, 12, 1950.0, 29, 7, -2350.0, 1950.0, 30, 12, 1950.0, 30, 12] 17
[-400.0, -6.779661016949152, 0, 59, 19, 30, 12, 1950.0, 29, 7, -2350.0, 1950.0, 30, 12, 1950.0, 30, 12] 17
[-400.0, -6.779661016949152, 0, 59, 19, 30, 12, 1950.0, 29, 7, -2350.0, 1950.0, 30, 12, 1950.0, 30, 12] 17
[-505.0, -7.537313432835821, 0, 67, 28, 34, 15, 2175.0, 33, 13, -2680.0, 2175.0, 34, 15, 2175.0, 34, 15] 17
[400.0, 6.153846153846154, 0, 65, 20, 33, 12, 2350.0, 32, 8, -1950.0, 2350.0, 33, 12, 2350.0, 33, 12] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,-400.0,-6.779661,0,59,19,30,12,1950.0,29,7,-2350.0,1950.0,30,12,Histogram a: 0.1
10,-400.0,-6.779661,0,59,19,30,12,1950.0,29,7,-2350.0,1950.0,30,12,Histogram a: 0.1
10,-400.0,-6.779661,0,59,19,30,12,1950.0,29,7,-2350.0,1950.0,30,12,Histogram a: 0.1
206,-505.0,-7.537313,0,67,28,34,15,2175.0,33,13,-2680.0,2175.0,34,15,Histogram a: -0.2 g: 800
9,400.0,6.153846,0,65,20,33,12,2350.0,32,8,-1950.0,2350.0,33,12,Histogram a: 0


In [8]:
for i in range(len(sys1_major)):
    sys1_major[i].extend(sys1_major[i][-3:])
    print(sys1_major[i], len(sys1_major[i]))

best_sys1_major = get_best_results(sys1_major)

[-1011.0, -4.156409268697399, 0, 177, 58, 89, 37, 6086.0, 88, 21, -7097.0, 6086.0, 89, 37, 6086.0, 89, 37] 17
[-104.0, 6.066172316384179, 0, 187, 59, 94, 34, 6528.0, 93, 25, -6632.0, 6528.0, 94, 34, 6528.0, 94, 34] 17
[2058.0, 43.56270009416197, 0, 193, 65, 97, 38, 7742.0, 96, 27, -5684.0, 7742.0, 97, 38, 7742.0, 97, 38] 17
[-769.0, -0.5625659580883395, 0, 187, 69, 94, 41, 6565.0, 93, 28, -7334.0, 6565.0, 94, 41, 6565.0, 94, 41] 17
[-66.0, -0.27868072787427334, 0, 187, 60, 94, 38, 6606.0, 93, 22, -6672.0, 6606.0, 94, 38, 6606.0, 94, 38] 17


# SYSTEM 2

In [9]:
signal_line_if = True
signal_line_if_target = False
sys2_major = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]

# first part

In [10]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys2_major,
    sys2_first,
    best_sys2_first,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys2_major,
    first_check,
    first_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[4600.5, 242.1315789473684, 0, 19, 13, 10, 7, 4469.5, 9, 6, 131.0, 4469.5, 10, 7, 4469.5, 10, 7] 17
[2658.5, 139.92105263157896, 0, 19, 13, 10, 7, 3498.5, 9, 6, -840.0, 3498.5, 10, 7, 3498.5, 10, 7] 17
[1328.5, 78.1470588235294, 0, 17, 12, 9, 6, 2833.5, 8, 6, -1505.0, 2833.5, 9, 6, 2833.5, 9, 6] 17
[1328.5, 78.1470588235294, 0, 17, 12, 9, 6, 2833.5, 8, 6, -1505.0, 2833.5, 9, 6, 2833.5, 9, 6] 17
[4074.5, 214.44736842105263, 0, 19, 14, 10, 8, 4527.5, 9, 6, -453.0, 4527.5, 10, 8, 4527.5, 10, 8] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
235,4600.5,242.131579,0,19,13,10,7,4469.5,9,6,131.0,4469.5,10,7,Histogram a: 0 g: 1
15,2658.5,139.921053,0,19,13,10,7,3498.5,9,6,-840.0,3498.5,10,7,Histogram a: 0.6
263,1328.5,78.147059,0,17,12,9,6,2833.5,8,6,-1505.0,2833.5,9,6,Histogram a: 0.1 g: 20
264,1328.5,78.147059,0,17,12,9,6,2833.5,8,6,-1505.0,2833.5,9,6,Histogram a: 0.1 g: 30
9,4074.5,214.447368,0,19,14,10,8,4527.5,9,6,-453.0,4527.5,10,8,Histogram a: 0


# second part

In [11]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys2_major,
    sys2_second,
    best_sys2_second,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys2_major,
    second_check,
    second_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[8696.0, 434.8, 0, 20, 12, 10, 7, 4882.0, 10, 5, 3814.0, 4882.0, 10, 7, 4882.0, 10, 7] 17
[6742.0, 321.04761904761904, 0, 21, 10, 11, 5, 4341.5, 10, 5, 2400.5, 4341.5, 11, 5, 4341.5, 11, 5] 17
[6742.0, 321.04761904761904, 0, 21, 10, 11, 5, 4341.5, 10, 5, 2400.5, 4341.5, 11, 5, 4341.5, 11, 5] 17
[8790.0, 439.5, 0, 20, 11, 10, 6, 4850.0, 10, 5, 3940.0, 4850.0, 10, 6, 4850.0, 10, 6] 17
[5462.5, 273.125, 0, 20, 10, 10, 6, 3335.0, 10, 4, 2127.5, 3335.0, 10, 6, 3335.0, 10, 6] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
211,8696.0,434.800000,0,20,12,10,7,4882.0,10,5,3814.0,4882.0,10,7,Histogram a: -0.1 g: 1
0,6742.0,321.047619,0,21,10,11,5,4341.5,10,5,2400.5,4341.5,11,5,Histogram a: -0.9
0,6742.0,321.047619,0,21,10,11,5,4341.5,10,5,2400.5,4341.5,11,5,Histogram a: -0.9
37,8790.0,439.500000,0,20,11,10,6,4850.0,10,5,3940.0,4850.0,10,6,Histogram a: -0.9 g: 700
9,5462.5,273.125000,0,20,10,10,6,3335.0,10,4,2127.5,3335.0,10,6,Histogram a: 0


# third part

In [12]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys2_major,
    sys2_third,
    best_sys2_third,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys2_major,
    third_check,
    third_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[4735.0, 315.6666666666667, 0, 15, 11, 8, 6, 3870.0, 7, 5, 865.0, 3870.0, 8, 6, 3870.0, 8, 6] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[5025.0, 335.0, 0, 15, 12, 8, 6, 4015.0, 7, 6, 1010.0, 4015.0, 8, 6, 4015.0, 8, 6] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,4735.0,315.666667,0,15,11,8,6,3870.0,7,5,865.0,3870.0,8,6,Histogram a: 0.1
43,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 1
43,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 1
58,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 400
9,5025.0,335.000000,0,15,12,8,6,4015.0,7,6,1010.0,4015.0,8,6,Histogram a: 0


In [13]:
for i in range(len(sys2_major)):
    sys2_major[i].extend(sys2_major[i][-3:])
    print(sys2_major[i], len(sys2_major[i]))

best_sys2_major = get_best_results(sys2_major)

[18031.5, 992.598245614035, 0, 54, 36, 28, 20, 13221.5, 26, 16, 4810.0, 13221.5, 28, 20, 13221.5, 28, 20] 17
[13825.5, 755.968671679198, 0, 55, 31, 29, 17, 11395.0, 26, 14, 2430.5, 11395.0, 29, 17, 11395.0, 29, 17] 17
[12495.5, 694.1946778711484, 0, 53, 30, 28, 16, 10730.0, 25, 14, 1765.5, 10730.0, 28, 16, 10730.0, 28, 16] 17
[14543.5, 812.6470588235294, 0, 52, 31, 27, 17, 11238.5, 25, 14, 3305.0, 11238.5, 27, 17, 11238.5, 27, 17] 17
[14562.0, 822.5723684210526, 0, 54, 36, 28, 20, 11877.5, 26, 16, 2684.5, 11877.5, 28, 20, 11877.5, 28, 20] 17


# SYSTEM 3

In [14]:
signal_line_if = False
signal_line_if_target = True
sys3_major = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]

# first part

In [15]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys3_major,
    sys3_first,
    best_sys3_first,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys3_major,
    first_check,
    first_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[2658.5, 139.92105263157896, 0, 19, 13, 10, 7, 3498.5, 9, 6, -840.0, 3498.5, 10, 7, 3498.5, 10, 7] 17
[2658.5, 139.92105263157896, 0, 19, 13, 10, 7, 3498.5, 9, 6, -840.0, 3498.5, 10, 7, 3498.5, 10, 7] 17
[4642.5, 244.3421052631579, 0, 19, 13, 10, 7, 4490.5, 9, 6, 152.0, 4490.5, 10, 7, 4490.5, 10, 7] 17
[4642.5, 244.3421052631579, 0, 19, 13, 10, 7, 4490.5, 9, 6, 152.0, 4490.5, 10, 7, 4490.5, 10, 7] 17
[4074.5, 214.44736842105263, 0, 19, 14, 10, 8, 4527.5, 9, 6, -453.0, 4527.5, 10, 8, 4527.5, 10, 8] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
15,2658.5,139.921053,0,19,13,10,7,3498.5,9,6,-840.0,3498.5,10,7,Histogram a: 0.6
15,2658.5,139.921053,0,19,13,10,7,3498.5,9,6,-840.0,3498.5,10,7,Histogram a: 0.6
263,4642.5,244.342105,0,19,13,10,7,4490.5,9,6,152.0,4490.5,10,7,Histogram a: 0.1 g: 20
273,4642.5,244.342105,0,19,13,10,7,4490.5,9,6,152.0,4490.5,10,7,Histogram a: 0.1 g: 300
9,4074.5,214.447368,0,19,14,10,8,4527.5,9,6,-453.0,4527.5,10,8,Histogram a: 0


# second part

In [16]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys3_major,
    sys3_second,
    best_sys3_second,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys3_major,
    second_check,
    second_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[8836.0, 401.6363636363636, 0, 22, 14, 11, 8, 4952.0, 11, 6, 3884.0, 4952.0, 11, 8, 4952.0, 11, 8] 17
[7996.5, 399.825, 0, 20, 11, 10, 6, 4602.0, 10, 5, 3394.5, 4602.0, 10, 6, 4602.0, 10, 6] 17
[8836.0, 401.6363636363636, 0, 22, 14, 11, 8, 4952.0, 11, 6, 3884.0, 4952.0, 11, 8, 4952.0, 11, 8] 17
[8696.0, 434.8, 0, 20, 12, 10, 7, 4882.0, 10, 5, 3814.0, 4882.0, 10, 7, 4882.0, 10, 7] 17
[5462.5, 273.125, 0, 20, 10, 10, 6, 3335.0, 10, 4, 2127.5, 3335.0, 10, 6, 3335.0, 10, 6] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
211,8836.0,401.636364,0,22,14,11,8,4952.0,11,6,3884.0,4952.0,11,8,Histogram a: -0.1 g: 1
3,7996.5,399.825000,0,20,11,10,6,4602.0,10,5,3394.5,4602.0,10,6,Histogram a: -0.6
211,8836.0,401.636364,0,22,14,11,8,4952.0,11,6,3884.0,4952.0,11,8,Histogram a: -0.1 g: 1
165,8696.0,434.800000,0,20,12,10,7,4882.0,10,5,3814.0,4882.0,10,7,Histogram a: -0.3 g: 4
9,5462.5,273.125000,0,20,10,10,6,3335.0,10,4,2127.5,3335.0,10,6,Histogram a: 0


# third part

In [17]:
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    
    sys3_major,
    sys3_third,
    best_sys3_third,
    
) = return_strategy_performance(
    nikkei_data,
    
    sys3_major,
    third_check,
    third_part,
    
    strategy,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)
dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[4735.0, 315.6666666666667, 0, 15, 11, 8, 6, 3870.0, 7, 5, 865.0, 3870.0, 8, 6, 3870.0, 8, 6] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[4425.0, 295.0, 0, 15, 8, 8, 5, 3555.0, 7, 3, 870.0, 3555.0, 8, 5, 3555.0, 8, 5] 17
[5025.0, 335.0, 0, 15, 12, 8, 6, 4015.0, 7, 6, 1010.0, 4015.0, 8, 6, 4015.0, 8, 6] 17


,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
10,4735.0,315.666667,0,15,11,8,6,3870.0,7,5,865.0,3870.0,8,6,Histogram a: 0.1
43,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 1
43,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 1
63,4425.0,295.000000,0,15,8,8,5,3555.0,7,3,870.0,3555.0,8,5,Histogram a: -0.8 g: 900
9,5025.0,335.000000,0,15,12,8,6,4015.0,7,6,1010.0,4015.0,8,6,Histogram a: 0


In [18]:
for i in range(len(sys3_major)):
    sys3_major[i].extend(sys3_major[i][-3:])
    print(sys3_major[i], len(sys3_major[i]))

best_sys3_major = get_best_results(sys3_major)

[16229.5, 857.2240829346092, 0, 56, 38, 29, 21, 12320.5, 27, 17, 3909.0, 12320.5, 29, 21, 12320.5, 29, 21] 17
[15080.0, 834.746052631579, 0, 54, 32, 28, 18, 11655.5, 26, 14, 3424.5, 11655.5, 28, 18, 11655.5, 28, 18] 17
[17903.5, 940.9784688995215, 0, 56, 35, 29, 20, 12997.5, 27, 15, 4906.0, 12997.5, 29, 20, 12997.5, 29, 20] 17
[17763.5, 974.1421052631579, 0, 54, 33, 28, 19, 12927.5, 26, 14, 4836.0, 12927.5, 28, 19, 12927.5, 28, 19] 17
[14562.0, 822.5723684210526, 0, 54, 36, 28, 20, 11877.5, 26, 16, 2684.5, 11877.5, 28, 20, 11877.5, 28, 20] 17


# tables

In [19]:
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_first, sys2_first, sys3_first]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/nikkei_rules_{strategy}_first_part_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                 
System 1 Rule 1  2912.5            52      42.31       4157.5   
         Rule 2  1097.5            48      37.50       3197.5   
         Rule 3  3259.5            54      44.44       4411.5   
         GA      3259.5            54      44.44       4411.5   
         MACD    2015.5            60      35.00       3710.5   
System 2 Rule 1  4600.5            19      68.42       4469.5   
         Rule 2  2658.5            19      68.42       3498.5   
         Rule 3  1328.5            17      70.59       2833.5   
         GA      1328.5            17      70.59       2833.5   
         MACD    4074.5            19      73.68       4527.5   
System 3 Rule 1  2658.5            19      68.42       3498.5   
         Rule 2  2658.5            19      68.42       3498.5   
         Rule 3  4642.5            19      68.42       4490.5   
         GA      4642.5            19      68.42       4490.5   
         MACD    4074.5            19      73.68       4527.5   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 26           50.00       -1245.0   
         Rule 2                 24           41.67       -2100.0   
         Rule 3                 27           51.85       -1152.0   
         GA                     27           51.85       -1152.0   
         MACD                   30           46.67       -1695.0   
System 2 Rule 1                 10           70.00         131.0   
         Rule 2                 10           70.00        -840.0   
         Rule 3                  9           66.67       -1505.0   
         GA                      9           66.67       -1505.0   
         MACD                   10           80.00        -453.0   
System 3 Rule 1                 10           70.00        -840.0   
         Rule 2                 10           70.00        -840.0   
         Rule 3                 10           70.00         152.0   
         GA                     10           70.00         152.0   
         MACD                   10           80.00        -453.0   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            34.62  
         Rule 2            33.33  
         Rule 3            37.04  
         GA                37.04  
         MACD              23.33  
System 2 Rule 1            66.67  
         Rule 2            66.67  
         Rule 3            75.00  
         GA                75.00  
         MACD              66.67  
System 3 Rule 1            66.67  
         Rule 2            66.67  
         Rule 3            66.67  
         GA                66.67  
         MACD              66.67

In [20]:
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_second, sys2_second, sys3_second]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/nikkei_rules_{strategy}_second_part_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                 
System 1 Rule 1 -3523.5            66      25.76        -21.5   
         Rule 2  -801.5            80      27.50       1380.5   
         Rule 3  -801.5            80      27.50       1380.5   
         GA     -3523.5            66      25.76        -21.5   
         MACD   -2481.5            62      30.65        545.5   
System 2 Rule 1  8696.0            20      60.00       4882.0   
         Rule 2  6742.0            21      47.62       4341.5   
         Rule 3  6742.0            21      47.62       4341.5   
         GA      8790.0            20      55.00       4850.0   
         MACD    5462.5            20      50.00       3335.0   
System 3 Rule 1  8836.0            22      63.64       4952.0   
         Rule 2  7996.5            20      55.00       4602.0   
         Rule 3  8836.0            22      63.64       4952.0   
         GA      8696.0            20      60.00       4882.0   
         MACD    5462.5            20      50.00       3335.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 33           36.36       -3502.0   
         Rule 2                 40           30.00       -2182.0   
         Rule 3                 40           30.00       -2182.0   
         GA                     33           36.36       -3502.0   
         MACD                   31           38.71       -3027.0   
System 2 Rule 1                 10           70.00        3814.0   
         Rule 2                 11           45.45        2400.5   
         Rule 3                 11           45.45        2400.5   
         GA                     10           60.00        3940.0   
         MACD                   10           60.00        2127.5   
System 3 Rule 1                 11           72.73        3884.0   
         Rule 2                 10           60.00        3394.5   
         Rule 3                 11           72.73        3884.0   
         GA                     10           70.00        3814.0   
         MACD                   10           60.00        2127.5   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            15.15  
         Rule 2            25.00  
         Rule 3            25.00  
         GA                15.15  
         MACD              22.58  
System 2 Rule 1            50.00  
         Rule 2            50.00  
         Rule 3            50.00  
         GA                50.00  
         MACD              40.00  
System 3 Rule 1            54.55  
         Rule 2            50.00  
         Rule 3            54.55  
         GA                50.00  
         MACD              40.00

In [21]:
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_third, sys2_third, sys3_third]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/nikkei_rules_{strategy}_third_part_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                 
System 1 Rule 1  -400.0            59      32.20       1950.0   
         Rule 2  -400.0            59      32.20       1950.0   
         Rule 3  -400.0            59      32.20       1950.0   
         GA      -505.0            67      41.79       2175.0   
         MACD     400.0            65      30.77       2350.0   
System 2 Rule 1  4735.0            15      73.33       3870.0   
         Rule 2  4425.0            15      53.33       3555.0   
         Rule 3  4425.0            15      53.33       3555.0   
         GA      4425.0            15      53.33       3555.0   
         MACD    5025.0            15      80.00       4015.0   
System 3 Rule 1  4735.0            15      73.33       3870.0   
         Rule 2  4425.0            15      53.33       3555.0   
         Rule 3  4425.0            15      53.33       3555.0   
         GA      4425.0            15      53.33       3555.0   
         MACD    5025.0            15      80.00       4015.0   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 30           40.00       -2350.0   
         Rule 2                 30           40.00       -2350.0   
         Rule 3                 30           40.00       -2350.0   
         GA                     34           44.12       -2680.0   
         MACD                   33           36.36       -1950.0   
System 2 Rule 1                  8           75.00         865.0   
         Rule 2                  8           62.50         870.0   
         Rule 3                  8           62.50         870.0   
         GA                      8           62.50         870.0   
         MACD                    8           75.00        1010.0   
System 3 Rule 1                  8           75.00         865.0   
         Rule 2                  8           62.50         870.0   
         Rule 3                  8           62.50         870.0   
         GA                      8           62.50         870.0   
         MACD                    8           75.00        1010.0   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            24.14  
         Rule 2            24.14  
         Rule 3            24.14  
         GA                39.39  
         MACD              25.00  
System 2 Rule 1            71.43  
         Rule 2            42.86  
         Rule 3            42.86  
         GA                42.86  
         MACD              85.71  
System 3 Rule 1            71.43  
         Rule 2            42.86  
         Rule 3            42.86  
         GA                42.86  
         MACD              85.71

In [22]:
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major, sys2_major, sys3_major]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nikkei/nikkei_rules_{strategy}_major_{params_str}.csv", index=True)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Rule 1  -1011.0           177      32.77       6086.0   
         Rule 2   -104.0           187      31.55       6528.0   
         Rule 3   2058.0           193      33.68       7742.0   
         GA       -769.0           187      36.90       6565.0   
         MACD      -66.0           187      32.09       6606.0   
System 2 Rule 1  18031.5            54      66.67      13221.5   
         Rule 2  13825.5            55      56.36      11395.0   
         Rule 3  12495.5            53      56.60      10730.0   
         GA      14543.5            52      59.62      11238.5   
         MACD    14562.0            54      66.67      11877.5   
System 3 Rule 1  16229.5            56      67.86      12320.5   
         Rule 2  15080.0            54      59.26      11655.5   
         Rule 3  17903.5            56      62.50      12997.5   
         GA      17763.5            54      61.11      12927.5   
         MACD    14562.0            54      66.67      11877.5   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Rule 1                 89           41.57       -7097.0   
         Rule 2                 94           36.17       -6632.0   
         Rule 3                 97           39.18       -5684.0   
         GA                     94           43.62       -7334.0   
         MACD                   94           40.43       -6672.0   
System 2 Rule 1                 28           71.43        4810.0   
         Rule 2                 29           58.62        2430.5   
         Rule 3                 28           57.14        1765.5   
         GA                     27           62.96        3305.0   
         MACD                   28           71.43        2684.5   
System 3 Rule 1                 29           72.41        3909.0   
         Rule 2                 28           64.29        3424.5   
         Rule 3                 29           68.97        4906.0   
         GA                     28           67.86        4836.0   
         MACD                   28           71.43        2684.5   

                 Short Win Ratio  
first    second                   
System 1 Rule 1            23.86  
         Rule 2            26.88  
         Rule 3            28.12  
         GA                30.11  
         MACD              23.66  
System 2 Rule 1            61.54  
         Rule 2            53.85  
         Rule 3            56.00  
         GA                56.00  
         MACD              61.54  
System 3 Rule 1            62.96  
         Rule 2            53.85  
         Rule 3            55.56  
         GA                53.85  
         MACD              61.54

In [23]:
# sys_modes = ["System 1", "System 2", "System 3"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
data = [
    [best_sys1_major, sys1_major[-1]],
    [best_sys2_major, sys2_major[-1]],
]

for sys_mode in sys_modes:
    for entry in entries:
        tuples.append((sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/nikkei/nikkei_rules_{strategy}_overall_major_{params_str}.csv", index=True
)

table_data

Return  Transactions  Win Ratio  Long Return  \
first    second                                                  
System 1 Best     2058.0           193      33.68       7742.0   
         MACD      -66.0           187      32.09       6606.0   
System 2 Best    18031.5            54      66.67      13221.5   
         MACD    14562.0            54      66.67      11877.5   

                 Long Transactions  Long Win Ratio  Short Return  \
first    second                                                    
System 1 Best                   97           39.18       -5684.0   
         MACD                   94           40.43       -6672.0   
System 2 Best                   28           71.43        4810.0   
         MACD                   28           71.43        2684.5   

                 Short Win Ratio  
first    second                   
System 1 Best              28.12  
         MACD              23.66  
System 2 Best              61.54  
         MACD              61.54